# Run the plotting notebooks

Executes the plotting notebooks in this folder (kernel `Python (Pulse)`) for the parts in `PARTS` and, optionally, only the figures in `ONLY`.

In [ ]:
from pathlib import Path
import nbformat
from nbconvert.preprocessors import ExecutePreprocessor

PARTS = ["main", "supplementary"]
ONLY = []

def run_notebook(folder, name):
    """Execute <folder>/<name>.ipynb in place."""
    if not (folder / "data").exists():
        return "no data"
    for old in folder.glob("*.png"):
        old.unlink()
    path = folder / f"{name}.ipynb"
    nb = nbformat.read(path, as_version=4)
    try:
        ExecutePreprocessor(timeout=3600, kernel_name="pulse").preprocess(
            nb, {"metadata": {"path": str(folder)}})
        status = f"ok ({len(list(folder.glob('*.png')))} png)"
    except Exception as e:
        status = "FAILED: " + str(e).strip().splitlines()[-1][:300]
    nbformat.write(nb, path)
    return status


ROOT = Path(".").resolve()
for part in PARTS:
    for folder in sorted(p for p in (ROOT / part).iterdir() if p.is_dir()):
        if ONLY and folder.name not in ONLY:
            continue
        print(f"{part:14s} {folder.name:18s} {run_notebook(folder, folder.name)}")